In [ ]:
import glob
import os

import matplotlib.pyplot as plt
import pandas as pd

import projects.minigrid_repro.analysis_utils as a_utils

experiment_dir = r"C:\Users\77019\Downloads\gradient-routing-main\gradient-routing-main\projects\minigrid_repro\data\oversight_levels"
experiment_dir = r"C:\Users\77019\Downloads\selected_folders_holdouts\data\oversight_levels"
experiment_dir = r"C:\Users\77019\Desktop\code\rl_ai_safety\gradient-routing\projects\minigrid_repro\data\1perc_all"
experiment_dir = r"C:\Users\77019\Desktop\code\rl_ai_safety\gradient-routing\projects\minigrid_repro\data\1perc_paper"
print("Reading files...", end=" ")
eval_files = glob.glob(os.path.join(experiment_dir, "eval_results*.csv"))
eval_dfs = [(pd.read_csv(file), file) for file in eval_files]

In [ ]:
    # if df["run_label"].iat[0] == "naive_outcomes+earlystop_0.002" and df['oversight_prob'].iat[0] == 0.008:
    # if df['run_label'][0] == "routing" and df['oversight_prob'][0] == 0.01:


In [ ]:
def get_avg_return(df):
    final_steps = (
        df[df.update_idx <= 20000]
        .sort_values("update_idx")
        .groupby(["run_label", "oversight_prob", "run_id"])
        .tail(1)
    )
    res = (
        final_steps.groupby(["run_label", "oversight_prob"])
        .agg({"avg_return": ["mean", a_utils.ci_width]})
        .reset_index()
    )
    return abs(res['avg_return']['mean'][0])

In [ ]:
names = []
for df, name in eval_dfs:
    if df['run_label'][0] == "routing+baseline" and df['oversight_prob'][0] == 0.01:
        # if get_avg_return(df) >= 0.2:
        # if df['update_idx'].max() < 10000:
            names.append(name.split('\\')[-1])

names.sort()

In [ ]:
names

In [ ]:
import os
import shutil   # <-- for file copying

# ── 1. Decide where you want the copies ─────────────────────────────
dest_dir = r"C:\Users\77019\Downloads\oversight_selected"
os.makedirs(dest_dir, exist_ok=True)          # makes it if it isn’t there

# ── 2. Copy each file we filtered for ───────────────────────────────
for df, src_path in eval_dfs:
    if df["run_label"].iat[0] == "naive_outcomes+earlystop_0.002" and df['oversight_prob'].iat[0] == 0.008:
        shutil.copy2(src_path,                    # source file (full path)
                     os.path.join(dest_dir,
                                  os.path.basename(src_path)))  # keep original name
        print(f"Copied {os.path.basename(src_path)} → {dest_dir}")


In [ ]:
import pandas as pd
profs_df = pd.read_csv(r"C:\Users\77019\Downloads\AI Safety Professionals.csv")

In [ ]:
type(profs_df[profs_df['Taking students?'].notna()]['Taking students?'][10])

In [ ]:
my_profs_df = profs_df[  (profs_df['Taking students?'] == 'checked') 
                       & (profs_df['Open Positions'])] 
my_profs_df = profs_df[profs_df['Taking students?'] == 'checked'] 

In [ ]:
my_profs_df

проверить графики холдаута на рандом ранах

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import glob
import os
import projects.minigrid_repro.analysis_utils as a_utils

import glob
import pandas as pd

experiment_dir = r"C:\Users\77019\Desktop\code\rl_ai_safety\gradient-routing\projects\minigrid_repro\data\rnd_hps"
experiment_dir = r"C:\Users\77019\Downloads\selected_folders_rnd_14\data\oversight_levels"
description = experiment_name = "rnd_hp"
smooth_amt = 1

import os
import pandas as pd
import glob
import matplotlib.pyplot as plt

# Load all train_results
train_files = glob.glob(os.path.join(experiment_dir, "train_results*.csv"))
train_res = pd.concat([pd.read_csv(f) for f in train_files])

# Load all eval_results
eval_files = glob.glob(os.path.join(experiment_dir, "eval_results*.csv"))
eval_res = pd.concat([pd.read_csv(f) for f in eval_files])

holdout_files = glob.glob(os.path.join(experiment_dir, "holdout_results*.csv"))
holdout_res = pd.concat([pd.read_csv(f) for f in holdout_files])

metrics_oversight = {}
for level in eval_res.oversight_prob.unique():
    metrics_oversight[level] = []

# Get all unique run_ids
run_ids = sorted(train_res.run_id.unique())

# Figure layout: rows = n_runs, cols = 3 (train, holdout, test)
n_runs = len(run_ids)
figsize = (12, 4 * n_runs)
fig, axes = plt.subplots(nrows=n_runs, ncols=3, figsize=figsize)
fig.suptitle(f"{description} ({n_runs} total runs)")

# metrics_oversight = 

# Loop through runs
for i, run_id in enumerate(run_ids):
    # TRAIN
    ax_train = axes[i, 0] if n_runs > 1 else axes[0]
    subset_train = train_res[train_res.run_id == run_id]
    a_utils.gplot(
        subset_train,
        x="update_idx",
        y="avg_return",
        group="run_label",
        smooth=smooth_amt,
        ax=ax_train,
    )
    ax_train.set_title(f"Run {run_id} - Train")
    ax_train.set_xlabel("Update step")
    ax_train.set_ylabel("Train Return")

    # HOLDOUT
    subset_holdout = holdout_res[holdout_res.run_id == run_id]
    ax_holdout = axes[i, 1] if n_runs > 1 else axes[1]
    a_utils.gplot(
        subset_holdout,
        x="update_idx",
        y="avg_return",
        group="run_label",
        smooth=smooth_amt,
        ax=ax_holdout,
    )
    ax_holdout.set_title(f"Run {run_id} - Holdout")
    ax_holdout.set_xlabel("Update step")
    ax_holdout.set_ylabel("Holdout Return")

    # EVAL
    ax_eval = axes[i, 2] if n_runs > 1 else axes[2]
    subset_eval = eval_res[eval_res.run_id == run_id]
    a_utils.gplot(
        subset_eval,
        x="update_idx",
        y="avg_return",
        group="run_label",
        smooth=smooth_amt,
        ax=ax_eval,
    )
    ax_eval.set_title(f"Run {run_id} - Eval")
    ax_eval.set_xlabel("Update step")
    ax_eval.set_ylabel("Eval Return")

    best_holdout_update_idx = subset_holdout.loc[subset_holdout["avg_return"].idxmax()]["update_idx"]
    ax_holdout.axvline(best_holdout_update_idx, color='red', linestyle='--', label='Best Update')

    metrics_oversight[subset_eval.oversight_prob.iloc[0]].append(subset_eval[subset_eval.update_idx == best_holdout_update_idx].avg_return.values[0])
# plt.tight_layout()
# plt.savefig(
#     os.path.join(
#         figures_dir,
#         f"rl_per_run_curves_{training_method}_{oversight_percent}.pdf",
#     ),
#     bbox_inches="tight",
# )



In [ ]:
for ovs_porb, metrics in metrics_oversight.items():
    print(ovs_porb, sum(metrics)/len(metrics))

In [ ]:
import glob, os
import pandas as pd

# 1) Load eval & holdout
eval_res    = pd.concat([pd.read_csv(f) for f in glob.glob(os.path.join(experiment_dir,"eval_results*.csv"))])
holdout_res = pd.concat([pd.read_csv(f) for f in glob.glob(os.path.join(experiment_dir,"holdout_results*.csv"))], ignore_index=True)

# 2) Merge in each run’s best‐holdout update
best_idx = (
    holdout_res
      .loc[holdout_res.groupby("run_id")["avg_return"].idxmax(),
           ["run_id","update_idx"]]
      .rename(columns={"update_idx":"best_update"})
)
eval_res = (
    eval_res
      .merge(best_idx, on="run_id", how="left")
      .query("best_update.isna() or update_idx <= best_update")
      .drop(columns=["best_update"])
)

# 3) Final‐step selection (one row per run)
final_steps = (
    eval_res
      .sort_values("update_idx")
      .groupby(["run_label","oversight_prob","run_id"], as_index=False)
      .tail(1)
)

# 4) Compute the oversight‐level means
means = final_steps.groupby("oversight_prob")["avg_return"].mean()
print(means)


In [ ]:
best_idx = pd.DataFrame(holdout_res.groupby("run_id")["avg_return"].idxmax()).rename(columns={"update_idx": "best_update"})

In [ ]:
holdout_res

In [ ]:
best_indexes = holdout_res.groupby("run_id")["avg_return"].idxmax()
best_rows = holdout_res.loc[best_indexes, ["run_id", "update_idx"]]
best_rows = best_rows.reset_index(drop=True)


In [ ]:
best_indexes

making holdouts from old verson

In [11]:
import pandas as pd
import matplotlib.pyplot as plt
import glob
import os
import projects.minigrid_repro.analysis_utils as a_utils

import glob
import pandas as pd

experiment_dir = r"C:\Users\77019\Desktop\code\rl_ai_safety\gradient-routing\projects\minigrid_repro\data\rnd_hps"
description = experiment_name = "rnd_hp"
smooth_amt = 1

import os
import pandas as pd
import glob
import matplotlib.pyplot as plt

# Load all train_results
train_files = glob.glob(os.path.join(experiment_dir, "train_results*.csv"))
train_res = pd.concat([pd.read_csv(f) for f in train_files])

# Load all eval_results
eval_files = glob.glob(os.path.join(experiment_dir, "eval_results*.csv"))
eval_res = pd.concat([pd.read_csv(f) for f in eval_files])

In [ ]:
# holdout_files = glob.glob(os.path.join(experiment_dir, "holdout_results*.csv"))
# holdout_res = pd.concat([pd.read_csv(f) for f in holdout_files])

# metrics_oversight = {}
# for level in eval_res.oversight_prob.unique():
#     metrics_oversight[level] = []

# Get all unique run_ids
run_ids = sorted(train_res.run_id.unique())

# Figure layout: rows = n_runs, cols = 3 (train, holdout, test)
n_runs = len(run_ids)
figsize = (12, 4 * n_runs)
fig, axes = plt.subplots(nrows=n_runs, ncols=3, figsize=figsize)
fig.suptitle(f"{description} ({n_runs} total runs)")

# metrics_oversight = 

# Loop through runs
for i, run_id in enumerate(run_ids):
    # TRAIN
    ax_train = axes[i, 0] if n_runs > 1 else axes[0]
    subset_train = train_res[train_res.run_id == run_id]
    a_utils.gplot(
        subset_train,
        x="update_idx",
        y="avg_return",
        group="run_label",
        smooth=smooth_amt,
        ax=ax_train,
    )
    ax_train.set_title(f"Run {run_id} - Train")
    ax_train.set_xlabel("Update step")
    ax_train.set_ylabel("Train Return")

    # HOLDOUT
    subset_holdout = holdout_res[holdout_res.run_id == run_id]
    ax_holdout = axes[i, 1] if n_runs > 1 else axes[1]
    a_utils.gplot(
        subset_holdout,
        x="update_idx",
        y="avg_return",
        group="run_label",
        smooth=smooth_amt,
        ax=ax_holdout,
    )
    ax_holdout.set_title(f"Run {run_id} - Holdout")
    ax_holdout.set_xlabel("Update step")
    ax_holdout.set_ylabel("Holdout Return")

    # EVAL
    ax_eval = axes[i, 2] if n_runs > 1 else axes[2]
    subset_eval = eval_res[eval_res.run_id == run_id]
    a_utils.gplot(
        subset_eval,
        x="update_idx",
        y="avg_return",
        group="run_label",
        smooth=smooth_amt,
        ax=ax_eval,
    )
    ax_eval.set_title(f"Run {run_id} - Eval")
    ax_eval.set_xlabel("Update step")
    ax_eval.set_ylabel("Eval Return")

    best_holdout_update_idx = subset_holdout.loc[subset_holdout["avg_return"].idxmax()]["update_idx"]
    ax_holdout.axvline(best_holdout_update_idx, color='red', linestyle='--', label='Best Update')

    metrics_oversight[subset_eval.oversight_prob.iloc[0]].append(subset_eval[subset_eval.update_idx == best_holdout_update_idx].avg_return.values[0])
# plt.tight_layout()
# plt.savefig(
#     os.path.join(
#         figures_dir,
#         f"rl_per_run_curves_{training_method}_{oversight_percent}.pdf",
#     ),
#     bbox_inches="tight",
# )



multiple runs auto work 

In [20]:
combine = True

eval_files = glob.glob(os.path.join(experiment_dir, "eval_results*.csv"))
eval_dfs = [pd.read_csv(file) for file in eval_files]
eval_res = pd.concat(eval_dfs)



In [21]:
eval_res

,update_idx,run_label,oversight_prob,global_step,policy_type,avg_return,run_id
0,0,naive_outcomes+earlystop_0.005,0.005,16384,training_policy,-0.000592,195816
1,1,naive_outcomes+earlystop_0.005,0.005,32768,training_policy,0.018887,195816
2,2,naive_outcomes+earlystop_0.005,0.005,49152,training_policy,-0.007376,195816
3,3,naive_outcomes+earlystop_0.005,0.005,65536,training_policy,0.014787,195816
4,4,naive_outcomes+earlystop_0.005,0.005,81920,training_policy,0.000077,195816
...,...,...,...,...,...,...,...
4628,4628,naive_outcomes+earlystop_0.003,0.007,75841536,training_policy,0.160170,983194
4629,4629,naive_outcomes+earlystop_0.003,0.007,75857920,training_policy,0.156775,983194
4630,4630,naive_outcomes+earlystop_0.003,0.007,75874304,training_policy,0.151237,983194
4631,4631,naive_outcomes+earlystop_0.003,0.007,75890688,training_policy,0.158140,983194


In [22]:

# --- Optional oversight_prob augmentation ---
if combine:

    def parse_extra_oversight(run_label):
        try:
            suffix = run_label.split("_")[-1]
            return float(suffix)
        except Exception:
            return 0.0  # fallback if malformed

    eval_res["oversight_holdout"] = eval_res["run_label"].apply(parse_extra_oversight)
    eval_res["oversight_prob"] += eval_res["oversight_holdout"]

    # rename run_label to show the fraction holdout/total
    def make_new_label(row):
        base = row["run_label"].rsplit("_", 1)[0]  # remove numeric suffix
        denom = row["oversight_prob"]
        num = row["oversight_holdout"]
        frac = round(num / denom, 1) if denom > 0 else 0.0
        return f"{base}_{frac}_frac"

    eval_res["run_label"] = eval_res.apply(make_new_label, axis=1)
    eval_res.drop(columns=["oversight_holdout"], inplace=True)

eval_res

,update_idx,run_label,oversight_prob,global_step,policy_type,avg_return,run_id
0,0,naive_outcomes+earlystop_0.5_frac,0.01,16384,training_policy,-0.000592,195816
1,1,naive_outcomes+earlystop_0.5_frac,0.01,32768,training_policy,0.018887,195816
2,2,naive_outcomes+earlystop_0.5_frac,0.01,49152,training_policy,-0.007376,195816
3,3,naive_outcomes+earlystop_0.5_frac,0.01,65536,training_policy,0.014787,195816
4,4,naive_outcomes+earlystop_0.5_frac,0.01,81920,training_policy,0.000077,195816
...,...,...,...,...,...,...,...
4628,4628,naive_outcomes+earlystop_0.3_frac,0.01,75841536,training_policy,0.160170,983194
4629,4629,naive_outcomes+earlystop_0.3_frac,0.01,75857920,training_policy,0.156775,983194
4630,4630,naive_outcomes+earlystop_0.3_frac,0.01,75874304,training_policy,0.151237,983194
4631,4631,naive_outcomes+earlystop_0.3_frac,0.01,75890688,training_policy,0.158140,983194


In [ ]:
python_path = r'c:\users\77019\pyver\py312\python.exe'
exp_name = 'rnd_paper_split'

!{python_path} -m projects.minigrid_repro.analyze_multiple_runs --subset_to_oversight 0.00000000001

Reading files... subsetting training points... done.
No data for oversight prob 1e-11.


c:\Users\77019\Desktop\code\rl_ai_safety\gradient-routing\projects\minigrid_repro\analyze_multiple_runs.py:123: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax.legend(bbox_to_anchor=(1.05, 0.5), loc="center left", fontsize=fontsize - 1)
